In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.5 Output head and weight tying

The final LayerNorm + linear head projects the hidden state to one logit per
vocab entry. PocketLM **ties** that head to the input embedding: one shared
`[vocab, n_embd]` matrix, fewer parameters, usually better. This completes
PocketLM v1, the model you train for the rest of the course.

In [ ]:
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
print("head tied to embedding:", model.head.weight is model.tok_emb.weight)
print("parameters:", sum(p.numel() for p in model.parameters()))
print(generate(model, tok, "The ", max_new_tokens=40, seed=0))

That is the full architecture: tokenize, embed, add positions, stack blocks,
norm, project to logits. Everything after this unit is about *training* it well
(u5), *modernizing* it (u6), *adapting* it (u7), and *serving* it (u8).

In [ ]:
assert model.head.weight is model.tok_emb.weight
assert len(generate(model, tok, "The ", max_new_tokens=40, seed=0)) == 4 + 40